In [12]:
import pandas as pd
import numpy as np
import os
import statsmodels.formula.api as smf

from statsmodels.genmod.bayes_mixed_glm import BinomialBayesMixedGLM

In [13]:
dirname = "data-files"

result_files = []
for files in os.listdir(dirname):
    if files.endswith(".csv"):
        result_files.append(f"{dirname}/{files}")

    else:
        continue

result_files

['data-files/sat_results-yoyo.csv',
 'data-files/sat_results-anna.csv',
 'data-files/sat_results-qian.csv']

In [14]:
all_data_list = []
for f in result_files:
    all_data_list.append(pd.read_csv(f))

all_data = pd.concat(all_data_list, ignore_index = True)
    
all_data

,timestamp,session_id,question_number,session_half,section,level,question,chosen_answer,correct_answer,correct,seconds_to_answer,video_setting,condition_order,video_on,video_loaded,video_mode
0,2026-11-06T12:21:48,20261106-122036-346,1,first,Algebra,easy,"If $3x + 2y = 10$ and $x - 2y = 6$, what is th...",4,4,1,71.80,follow_difficulty,NaN,1,1,video
1,2026-11-06T12:23:03,20261106-122036-346,2,first,Information and Ideas,hard,The author's main point about the relationship...,technology has had a mixed impact on the evolu...,technology has had a mixed impact on the evolu...,1,73.34,follow_difficulty,NaN,0,1,pause
2,2026-11-06T12:23:12,20261106-122036-346,3,first,Algebra,easy,"If $x = 3$ and $y = 2$, what is the value of $...",13,13,1,7.62,follow_difficulty,NaN,1,1,video
3,2026-11-06T12:24:39,20261106-122036-346,4,first,Craft and Structure,easy,"The author mentions Carlyle’s work as a poet, ...",suggest that Carlyle’s historical work was inf...,emphasize the range and complexity of Carlyle’...,0,85.58,follow_difficulty,NaN,1,1,video
4,2026-11-06T12:25:39,20261106-122036-346,5,first,Craft and Structure,easy,"In this passage, the author uses imagery to",help the reader visualize the experience.,help the reader visualize the experience.,1,55.33,follow_difficulty,NaN,1,1,video
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
78,2026-11-06T12:44:36,20261106-122205-557,10,first,Standard English Conventions,easy,Which of the following sentences correctly use...,"The small town, nestled in the valley, was kno...","The small town, nestled in the valley, was kno...",1,38.98,half_on_half_off_random_order,off_then_on,0,1,off
79,2026-11-06T12:45:12,20261106-122205-557,11,first,Advanced Math,hard,A circle has a radius of 5 centimeters. What i...,25\pi,25\pi,1,32.80,half_on_half_off_random_order,off_then_on,0,1,off
80,2026-11-06T12:48:08,20261106-122205-557,12,first,Standard English Conventions,hard,Which choice best combines the sentences at th...,increasing sales by 20 percent and boosting br...,"increasing sales by 20 percent, boosting brand...",0,173.29,half_on_half_off_random_order,off_then_on,0,1,off
81,2026-11-06T12:49:40,20261106-122205-557,13,first,Advanced Math,intermediate,The function $f(x) = 3x^2 + 5x - 2$ is defined...,-4,8,0,78.90,half_on_half_off_random_order,off_then_on,0,1,off


In [15]:
summary = (
    all_data.groupby(["session_id", "video_on"])
    ["seconds_to_answer"]
    .agg(["mean", "median", "count"])
    .round(2)
)

summary

mean  median  count
session_id          video_on                       
20261105-174615-57d 0          52.59   39.52     22
                    1          52.18   55.06     15
20261106-122036-346 0          50.45   48.91     19
                    1          51.89   56.16     13
20261106-122205-557 0         111.60   58.94     14

## Linear Mixed Effects Model

For time spent per question

In [16]:
all_data.columns = all_data.columns.str.strip().str.lstrip("*")

# convert relevant variables
all_data ["seconds_to_answer"] = pd.to_numeric(
    all_data["seconds_to_answer"], errors="coerce"
)
all_data['video_on'] = pd.to_numeric(
    all_data["video_on"], errors="coerce"
)

# remove empty values
all_data = all_data.dropna(subset=[
    "seconds_to_answer", "video_on", "session_id", "level", "section", "session_half"
])
all_data = all_data[all_data["seconds_to_answer"] > 0].copy()

# log-transform completion time
all_data['log_time'] = np.log(all_data["seconds_to_answer"])

# fit the linear mixed-effects model
model = smf.mixedlm(
    "log_time ~ video_on + C(level) + C(section) + C(session_half)",
    data=all_data,
    groups=all_data["session_id"]
)

result = model.fit(reml=False)

print(result.summary())

effect = np.exp(result.params["video_on"])

print(f"Video-on/off time ratio: {effect:.3f}")
print(f"Percent change: {(effect - 1) * 100:.1f}%")

                          Mixed Linear Model Regression Results
Model:                        MixedLM             Dependent Variable:             log_time
No. Observations:             83                  Method:                         ML      
No. Groups:                   3                   Scale:                          0.4388  
Min. group size:              14                  Log-Likelihood:                 -86.2595
Max. group size:              37                  Converged:                      Yes     
Mean group size:              27.7                                                        
------------------------------------------------------------------------------------------
                                                Coef.  Std.Err.   z    P>|z| [0.025 0.975]
------------------------------------------------------------------------------------------
Intercept                                        3.160    0.319  9.894 0.000  2.534  3.786
C(level)[T.hard]          

Correctness

In [18]:
all_data.columns = all_data.columns.str.strip().str.lstrip("*")

# convert relevant variables
all_data ["correct"] = pd.to_numeric(
    all_data["correct"], errors="coerce"
)
all_data['video_on'] = pd.to_numeric(
    all_data["video_on"], errors="coerce"
)

# remove empty values
all_data = all_data.dropna(subset=[
    "correct", "video_on", "session_id", "level", "section", "session_half"
]).copy()

# keep binary correct(1) and incorrect(0) answers
all_data = all_data[
    all_data["correct"].isin([0, 1]) & all_data["video_on"].isin([0, 1])
].copy()

all_data['correct'] = all_data["correct"].astype(int)
all_data['video_on'] = all_data["video_on"].astype(int)

# random intercept for each session
random_effects = {
    "session": "0 + C(session_id)"
}

# fit the logistic mixed-effects model
model = BinomialBayesMixedGLM.from_formula(
    "correct ~ video_on + C(level) + C(section) + C(session_half)",
    random_effects,
    all_data
)

result = model.fit_vb()

print(result.summary())

                                  Binomial Mixed GLM Results
                                                Type Post. Mean Post. SD   SD  SD (LB) SD (UB)
----------------------------------------------------------------------------------------------
Intercept                                          M     0.5852   0.2564                      
C(level)[T.hard]                                   M     0.8231   0.4805                      
C(level)[T.intermediate]                           M    -0.1659   0.4523                      
C(section)[T.Algebra]                              M    -0.0332   0.7072                      
C(section)[T.Craft and Structure]                  M    -0.7562   0.6037                      
C(section)[T.Expression of Ideas]                  M     0.9340   1.5898                      
C(section)[T.Geometry and Trigonometry]            M    -2.2912   0.8969                      
C(section)[T.Information and Ideas]                M    -0.1970   0.6133            

In [19]:
# extract video effect
video_idx = model.exog_names.index("video_on")

beta = result.fe_mean[video_idx]
se = result.fe_sd[video_idx]

# convert log-odds to odds ratio
odds_ratio = np.exp(beta)

# approximate 95% posterior credible interval
ci_lower = np.exp(beta - 1.96 * se)
ci_upper = np.exp(beta + 1.96 * se)

print(f"\nVideo-on/off odds ratio: {odds_ratio:.3f}")
print(f"Percent change in odds: {(odds_ratio - 1) * 100:.1f}%")
print(f"Approx. 95% credible interval: [{ci_lower:.3f}, {ci_upper:.3f}]")


Video-on/off odds ratio: 2.603
Percent change in odds: 160.3%
Approx. 95% credible interval: [1.100, 6.156]


## Accuracy by session

In [20]:
session_accuracy = (
    all_data.groupby(["session_id", "video_on"])["correct"]
    .agg(["mean", "sum", "count"])
)

session_accuracy["accuracy_percent"] = (
    session_accuracy["mean"] * 100
)

session_accuracy

mean  sum  count  accuracy_percent
session_id          video_on                                        
20261105-174615-57d 0         0.636364   14     22         63.636364
                    1         0.733333   11     15         73.333333
20261106-122036-346 0         0.631579   12     19         63.157895
                    1         0.692308    9     13         69.230769
20261106-122205-557 0         0.500000    7     14         50.000000

## Condition Order Effects

In [21]:
print(pd.crosstab(
    all_data["condition_order"],
    all_data["video_on"]
))

print(pd.crosstab(
    all_data["session_half"],
    all_data["video_on"]
))

video_on          0   1
condition_order        
off_then_on      21   0
on_then_off      15  15
video_on       0   1
session_half        
first         39  27
second        16   1
